# RAER — stage 00: pre-flight audit

Run this **before** any labeling or training. It answers the questions that decide
whether the dataset is usable at all: what actually ships, whether `P###` is a real
subject id, how the classes are distributed, whether an official split is included,
and how badly a naive split would leak subjects.

Two stages, cheapest first:

* **A — manifest only.** Reads Kaggle's file listing (names + sizes). Answers the
  clip, identity and leakage questions in about a minute, with **no download**.
* **B — full copy.** ~100 GB. Only worth it once stage A looks right.

See `DATA_PROVENANCE.md` for the licensing position. The Kaggle copy is an interim
source while official access is pending; nothing derived from it gets published.

## 1. Setup

In [ ]:
# Derived artifacts live on Drive so they survive the runtime dying.
# The MEDIA stays on local disk — it is large, slow over Drive, and must not be shared.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/RAER")       # derived artifacts only
LOCAL = Path("/content/raer")                      # media, never synced to Drive
for d in (DRIVE, LOCAL): d.mkdir(parents=True, exist_ok=True)
print("drive:", DRIVE, "| local:", LOCAL)
!df -h /content | tail -1

In [ ]:
# Gets RAER-external + LLMSTU-pipeline onto the runtime.
# Tries GitHub first; falls back to a zip in Drive, so this works before the
# RAER-external repo exists on GitHub.
RAER_REPO   = "vaelkokach/RAER-external"
LLMSTU_REPO = "vaelkokach/LLMSTU-pipeline"
DRIVE_ZIP   = "/content/drive/MyDrive/RAER/RAER-external.zip"   # fallback

import os, sys, subprocess, zipfile
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return os.environ.get(n)

gh = _secret("GITHUB_TOKEN")

def have(name): return os.path.isdir(f"/content/{name}")

def clone(repo):
    name = repo.split("/")[-1]
    if have(name):
        return True
    auth = f"{gh}@" if gh else ""
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{repo}.git",
                        f"/content/{name}"], capture_output=True, text=True)
    if r.returncode:
        print(f"[setup] clone {repo} failed: {r.stderr.strip().splitlines()[-1:] or r.returncode}")
    return r.returncode == 0

clone(LLMSTU_REPO)
if not clone(RAER_REPO):
    if not os.path.isdir("/content/drive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception:
            pass
    if os.path.exists(DRIVE_ZIP):
        print(f"[setup] falling back to {DRIVE_ZIP}")
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            z.extractall("/content")
    else:
        raise SystemExit(
            "RAER-external is not on GitHub and no zip at " + DRIVE_ZIP + ".\n"
            "Either create the repo and push, or drop RAER-external.zip in "
            "MyDrive/RAER/ and re-run this cell.")

assert have("RAER-external") and have("LLMSTU-pipeline"), "setup failed"
sys.path[:0] = ["/content/RAER-external", "/content/LLMSTU-pipeline"]
os.chdir("/content/RAER-external")
LLMSTU = "/content/LLMSTU-pipeline"
try:
    print("llmstu-pipeline @", subprocess.check_output(
        ["git", "-C", LLMSTU, "rev-parse", "--short", "HEAD"], text=True).strip())
except Exception:
    pass

In [ ]:
!pip install -q kaggle pandas

In [ ]:
# Kaggle credentials — add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (key icon).
os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME") or ""
os.environ["KAGGLE_KEY"]      = _secret("KAGGLE_KEY") or ""
assert os.environ["KAGGLE_KEY"], "Set KAGGLE_USERNAME / KAGGLE_KEY in Colab Secrets"

DATASET = "bearmn/raer-video-emotion-dataset"

## 2. Stage A — audit the manifest (no download)

If the clip count is far off 2,649, or the identity verdict is not what we expect,
stop here rather than spending 100 GB and an hour finding out.

In [ ]:
from raer import kaggle_listing, audit

listing = kaggle_listing.fetch(DATASET)
kaggle_listing.save(listing, DRIVE / "audit" / "listing.json")
print(f"{len(listing):,} files, {sum(s for _, s in listing)/1e9:.1f} GB")

r = audit.run(listing=listing, out_dir=DRIVE / "audit", source=f"kaggle:{DATASET}")

In [ ]:
print("clips        ", r["clips"]["n_clips"], "vs paper", r["clips"]["paper_n_clips"],
      f"({r['clips']['delta_vs_paper']:+d})")
print("frames/clip  ", r["clips"]["frames_per_clip"])
print("identity     ", r["identity"]["verdict"])
print("split key    ", r["identity"]["safe_split_key"])
print("leakage      ", r["leakage"]["verdict"])

In [ ]:
from IPython.display import HTML
HTML((DRIVE / "audit" / "audit_report.html").read_text())

## 3. Fetch just the small metadata files

The label mapping and any split lists are a few MB. Pull only those, then re-run the
audit with them so the distribution and split checks can complete.

In [ ]:
# Only the annotation/bbox files — a few MB, not the 100 GB of media.
small = [p for p, s in listing
         if Path(p).suffix.lower() in {".json", ".txt", ".csv"} and s < 50_000_000]
print("
".join(small))

meta_dir = LOCAL / "meta"
kaggle_listing.download_files(DATASET, small, meta_dir)   # keeps full relative paths
!find {meta_dir} -type f | head -20

In [ ]:
# Annotations are RAER/annotation/{train,test}.txt in the format
#   <clip dir> <n_frames> <label idx 1-5>
# The index->class mapping is undocumented; find_labels recovers it by matching
# observed frequencies to the published distribution and prints the confidence.
labels = audit.find_labels(meta_dir)
splits = audit.find_splits(meta_dir)
print(f"labels recovered: {len(labels):,}   splits: {{k: len(v) for k, v in splits.items()}}")

dist = audit.class_distribution(labels)
print(dist["status"])
import pandas as pd
pd.DataFrame(dist["rows"])

In [ ]:
sc = audit.split_check(splits)
print(sc["verdict"])
print("clip overlap    :", sc["clip_overlaps"])
print("identity overlap:", sc["identity_overlaps"], sc["identity_overlap_pct"])

## 4. Stage B — full copy (only if stage A is clean)

~100 GB. Check free disk first: Colab gives ~235 GB with ~47 GB already used, so this
fits, but not alongside a second copy. Downloading unzipped avoids needing 2x the space.

In [ ]:
!df -h /content | tail -1
# uncomment to run — this takes a while
# !kaggle datasets download -d {DATASET} -p {LOCAL} --unzip

In [ ]:
# Full audit over the downloaded copy
# r_full = audit.run(root=LOCAL / "RAER", out_dir=DRIVE / "audit_full")
# HTML((DRIVE / "audit_full" / "audit_report.html").read_text())

## 5. Gate

Resolved by the 2026-08-28 run (see `DATA_PROVENANCE.md`):

- [x] clip count — annotation covers exactly 2,649; the 20 extra mp4 are unannotated, drop them
- [x] label map — `1 neutrality, 2 enjoyment, 3 confusion, 4 fatigue, 5 distraction`, confirmed to 0.00pp
- [x] class distribution — matches the published figures exactly; this is the full set, not a subset
- [x] subject identity — `P###` is session-local; split on `identity_key` (141 keys vs 136 subjects)
- [x] `output_annotation/*` are a third party's VLM captions, not RAER labels — ignore
- [x] **the shipped split leaks 93.6% of subjects across train/test — build our own**

Still open:

- [ ] official access request resolved, and every finding re-verified against that copy
- [ ] confusion has only 37 clips total — decide how (or whether) to report per-class numbers on it